# A Quick Recap for the each part of the AI agents field


##  LangChain vs LangGraph State & Component Comparison

This document summarizes the key concepts discussed: **LangChain**, **LangGraph**, **AgentState**, **TypedDict State**, **dataclass Context**, and **Pydantic BaseModel**.

---

### 1. High-Level Architecture

| Layer | Purpose | Typical Components | Who Owns It |
|------|---------|-------------------|-------------|
| Capability Layer | Provides AI functionality | LLMs, Tools, Prompts | LangChain |
| Execution Layer | Controls flow of execution | Graph nodes, edges, reducers | LangGraph |
| Agent Memory | Stores agent reasoning state | AgentState | LangChain |
| Workflow State | Data passed between steps | TypedDict State | LangGraph |
| Runtime Context | External configuration | dataclass context | Application |
| Data Contracts | Validate structured input/output | Pydantic BaseModel | Tools / APIs |

---

### 2. LangChain vs LangGraph

| Feature | LangChain | LangGraph |
|-------|-----------|-----------|
| Primary Role | AI capability framework | Execution orchestration |
| Core Concept | Agents | Graph workflows |
| Control Flow | Implicit agent loop | Explicit graph |
| State Management | Agent memory | Workflow state |
| Parallel Execution | Limited | Native |
| Determinism | Lower | High |
| Production Workflows | Harder to manage | Designed for production |
| Tooling | LLMs, prompts, tools | Nodes, edges, reducers |
| Interrupt Support | Agent-level | Graph-level |
| Checkpointing | Limited | Built-in |

---

### 3. AgentState vs TypedDict State

| Aspect | AgentState | TypedDict State |
|------|-------------|----------------|
| Library | LangChain | LangGraph |
| Purpose | Agent internal memory | Workflow shared state |
| Structure | Python class | Dictionary schema |
| Mutation Style | Mutable object | Functional updates |
| Scope | Single agent | Entire workflow |
| Supports Reducers | No | Yes |
| Parallel Updates | No | Yes |
| Context Growth | Continuous | Controlled |
| Used By | Tools, middleware | Graph nodes |
| Control Flow | Agent loop | Graph edges |

---

### 4. dataclass Context vs AgentState

| Aspect | dataclass Context | AgentState |
|------|------------------|-----------|
| Purpose | Runtime configuration | Agent memory |
| Mutability | Typically static | Mutable |
| Lifecycle | Provided at invocation | Changes during execution |
| Storage | Outside the agent | Inside the agent |
| Example Data | API keys, user info | authentication flags |
| Persistence | No | Optional |
| Context Window Impact | None | Yes |


### 5. TypedDict vs BaseModel

| Aspect | TypedDict | BaseModel |
|------|------------------|-----------|
| Library | Python typing | Pydantic |
| Purpose | Describe dictionary structure | Validate structured data |
| Runtime Validation | No | Yes |
| Used In | Graph state |Tools / APIs |
| Serialization | Manual | Automatic |
| Strict Typing | Static only | Runtime enforcedOptional |
| Ideal For | Workflow data flow | API schemas  |



### 6. State Evolution Comparison

| Property        | AgentState          | TypedDict         |
| --------------- | ------------------- | ----------------- |
| Memory Pattern  | Accumulating memory | Selective updates |
| Context Size    | Grows continuously  | Controlled        |
| Data Ownership  | Agent               | Workflow          |
| Update Method   | Attribute mutation  | Return dictionary |
| Parallel Safety | No                  | Yes               |


### 7. Mental Model Summary

| Concept           | Think of it as                       |
| ----------------- | ------------------------------------ |
| LangChain         | AI capability toolkit                |
| LangGraph         | Workflow execution engine            |
| AgentState        | What the agent remembers             |
| TypedDict State   | Data flowing through the workflow    |
| dataclass Context | Configuration provided to the system |
| BaseModel         | Contract enforcing structured data   |


### 8. Final Simplified Architecture Diagram
```text
Application
   |
   |-- Context (dataclass)
   |
LangGraph Workflow
   |
   |-- State (TypedDict)
   |
   |-- Node
        |
        |-- LangChain Agent
                |
                |-- AgentState
                |-- Tools
                |-- LLM
```

# RAG From Scratch: Routing

Routing directs a user query to the most appropriate data source or prompt before retrieval. This notebook covers two routing strategies:
1. **Logical Routing** — Uses LLM function-calling to classify the query into a predefined category (e.g., Python/JS/Go docs).
2. **Semantic Routing** — Embeds the query and candidate prompts, then routes to the prompt with highest cosine similarity.

![alt text](<../../assets/Logical and Semantic routing.png>)

### Environment Initialization

Loads `.env` variables for LangSmith tracing, Mistral API, HuggingFace token, and User-Agent. Suppresses warnings.

In [ ]:
import warnings
import os 
from dotenv import load_dotenv

# Suppress all warnings for cleaner notebook output
warnings.filterwarnings("ignore")

# Load environment variables from the .env file into the process
load_dotenv()

try: 
    # Map .env variables to the keys LangChain/LangSmith expects at runtime
    os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGSMITH_TRACING_V2")   # Enable LangSmith tracing
    os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")         # LangSmith authentication key
    os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT")         # LangSmith project name for grouping traces
    os.environ["LANGCHAIN_ENDPOINT"] = os.getenv("LANGSMITH_ENDPOINT")       # LangSmith API endpoint URL
    os.environ["MISTRAL_API_KEY"] = os.getenv("MISTRAL_API_KEY")             # Mistral AI LLM/embedding API key
    os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")                           # HuggingFace access token
    os.environ["USER_AGENT"] = "MyLangChainApp/1.0"                           # Required User-Agent header for WebBaseLoader
    print("Environment variables set successfully")
except Exception as e: 
    print(f"Error: {e}")

Environment variables set successfully


### Part 10: Logical Routing

Uses **structured output / function-calling** to classify a query into one of three datasources: `python_docs`, `js_docs`, or `golang_docs`.

- **`RouteQuery`** — A Pydantic `BaseModel` with a `Literal` field constraining the output to valid datasource names.
- **`llm.with_structured_output(RouteQuery)`** — Forces the LLM to return a valid `RouteQuery` object.
- **Router chain** — System prompt + human question → structured LLM → `RouteQuery` instance.

In [ ]:
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_mistralai import ChatMistralAI


# Pydantic schema for structured routing output
# The Literal type constrains the LLM to only return one of the three valid datasources
class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""
    datasource: Literal["python_docs", "js_docs", "golang_docs"] = Field(
        ...,                              # Required field (no default)
        description = "Given a user question choose which datasource would be most relevant for answering their question",
    )
    
# Initialize LLM and wrap it with structured output to enforce RouteQuery schema
llm = ChatMistralAI(model = "mistral-medium-latest", temperature=0)
structured_llm = llm.with_structured_output(RouteQuery)  # Forces LLM to return a RouteQuery object

# System prompt instructs the LLM to classify questions by programming language
system = """You are an expert at routing a user question to the appropriate data source.

Based on the programming language the question is referring to, route it to the relevant data source."""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),        # System instruction for routing behavior
        ("human", "{question}"),   # User question to classify
    ]
)

# Router chain: prompt → structured LLM → RouteQuery object
router = prompt | structured_llm

#### Test Logical Router

Invokes the router with a Python-related question. The LLM classifies it and returns a `RouteQuery` object. Access `result.datasource` to get the string classification (e.g., `"python_docs"`).

> **Note**: The LLM uses function calling to produce structured output — it doesn't just return text, it returns a validated Pydantic object.

In [ ]:
# Test question — a Python/LangChain code snippet that should route to "python_docs"
question =  """Why doesn't the following code work:

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(["human", "speak in {language}"])
prompt.invoke("french")
"""

# Invoke the router chain — returns a validated RouteQuery Pydantic object
result = router.invoke({"question": question})

result  # RouteQuery(datasource='python_docs')

RouteQuery(datasource='python_docs')

In [ ]:
# Access the datasource classification directly from the Pydantic object
result.datasource  # Returns e.g. 'python_docs'

'python_docs'

#### Branching with RunnableLambda

Uses `RunnableLambda(choose_route)` to map the `RouteQuery.datasource` value to the appropriate downstream chain. In production, each branch would contain the actual retrieval/RAG logic for that datasource. Here it returns a placeholder string to demonstrate the routing.

In [ ]:
from langchain_core.runnables import RunnableLambda


def choose_route(result):
    """Map the RouteQuery.datasource value to the appropriate downstream chain.
    
    In production, each branch would contain the actual retrieval/RAG logic
    for that datasource. Here we return placeholder strings for demonstration.
    """
    if "python_docs" in result.datasource.lower():
        return "chain for python_docs"       # Placeholder — add Python docs RAG chain here
    if "js_docs" in result.datasource.lower():
        return "chain for js_docs"           # Placeholder — add JS docs RAG chain here
    else:
        return "chain for golang_docs"       # Placeholder — add Go docs RAG chain here

# Full routing chain: classify question → branch to appropriate chain
full_chain = router | RunnableLambda(choose_route)

full_chain.invoke({"question": question})

'chain for python_docs'

### Semantic Routing

Routes the query by **embedding similarity** rather than LLM classification.

- Two candidate prompts are defined (physics expert, math expert) and pre-embedded.
- `prompt_router()` embeds the user query, computes cosine similarity against each prompt embedding, and selects the most similar prompt template.
- The selected prompt feeds into the LLM to generate the answer.

This approach is faster and cheaper than function-calling routing since it avoids an extra LLM call for classification.

In [ ]:
from langchain_community.utils.math import cosine_similarity
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings

# Define two candidate prompts — one for physics, one for math
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{query}"""

math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{query}"""


# Pre-embed both prompt templates for similarity comparison
embeddings = MistralAIEmbeddings(model = "mistral-embed")
prompt_template = [physics_template, math_template]
prompt_embeddings = embeddings.embed_documents(prompt_template)  # Shape: (2, embedding_dim)

def prompt_router(input):
    """Route the query to the most semantically similar prompt template.
    
    Embeds the user query, computes cosine similarity against each candidate
    prompt embedding, and selects the prompt with the highest similarity score.
    """
    # Embed the user's query
    query_embedding = embeddings.embed_query(input["query"])
    
    # Compute cosine similarity between query and each prompt template
    similarity = cosine_similarity([query_embedding], prompt_embeddings)[0]
    
    # Select the prompt with the highest similarity score
    most_similar = prompt_template[similarity.argmax()]
    
    # Log which prompt was selected
    print("Using MATH" if most_similar == math_template else "Using PHYSICS")
    return PromptTemplate.from_template(most_similar)

# Semantic routing chain:
#   1. Package the query into a dict
#   2. Route to the best-matching prompt template
#   3. Pass through LLM and parse to string
chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | ChatMistralAI(model = "mistral-medium-latest", temperature = 0)
    | StrOutputParser()
)

# Test with a physics question — should route to the physics prompt
print(chain.invoke("What's a black hole"))

Using PHYSICS
A **black hole** is a region in space where gravity is so strong that nothing—not even light—can escape from it. Here’s a concise breakdown:

1. **Formation**: Most black holes form when massive stars collapse under their own gravity at the end of their life cycles (supernova explosion). Supermassive black holes (millions to billions of times the Sun’s mass) likely form differently, often at the centers of galaxies.

2. **Key Features**:
   - **Event Horizon**: The "point of no return" around a black hole. Once crossed, escape is impossible.
   - **Singularity**: At the center, matter is crushed into an infinitely dense point (where our current physics breaks down).
   - **No-Hair Theorem**: Black holes are described by just three properties: **mass**, **electric charge**, and **spin** (angular momentum). All other details ("hair") are lost.

3. **Why "Black"?**: Light can’t escape, so they’re invisible. We detect them by observing their effects on nearby matter (e.g., ac

**`RunnableLambda`**: The Custom Worker
- Use RunnableLambda when you want to run your own custom Python code inside a chain. It turns a regular function into a "Runnable" that LangChain understands.

  - Main Use: Formatting text, cleaning data, or calling an external API that doesn't have a built-in LangChain tool.

  - Example: Changing a user's input to all lowercase before sending it to the AI.

**`RunnablePassthrough`**: The Conveyor Belt
- Use RunnablePassthrough to pass data from one step to the next without changing it. It is most commonly used to "carry" the user's original question forward while you are looking up information in a database.

  - Main Use: Keeping the original input available for later steps in the chain (often used with `.assign()`).

  - Example: If you need to search a database using a question, but you also need that same question later to generate the final answer.


| Tool | Analogy | Best For
| ----------------- | ---------------|--------------------- |
| RunnableLambda    | A custom machine |  Transforming data or running custom logic.
| RunnablePassthrough | A clear conveyor belt| Passing data forward unchanged or adding new keys.